## Q4. Saving traces to SQLite

In [2]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

## # OpenTelemetry Load

In [3]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()

provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [4]:
from rag_helper import RAGBase, INSTRUCTIONS, PROMPT_TEMPLATE
from starter import index, client

class RAGTraced(RAGBase):

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search_operation") as span:
            return self.index.search(query, num_results=num_results)
        

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['filename'])
            lines.append(doc['content'])
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        with tracer.start_as_current_span("llm_operation") as span:
            input_messages = [
                {'role': 'developer', 'content': self.instructions},
                {'role': 'user', 'content': prompt}
            ]

            response = self.llm_client.responses.create(
                model=self.model,
                input=input_messages
            )

            input_cost = response.usage.input_tokens * (0.75 / 1_000_000)
            output_cost = response.usage.output_tokens * (4.50 / 1_000_000)
            cost = input_cost + output_cost

            span.set_attribute("input_tokens", response.usage.input_tokens)
            span.set_attribute("output_tokens", response.usage.output_tokens)
            span.set_attribute("cost", cost)
            
            return response
        
    

    def rag(self, query):
        with tracer.start_as_current_span("rag_operation") as span:
            search_results = self.search(query)
            prompt = self.build_prompt(query, search_results)
            response = self.llm(prompt)
            return response.output_text
    


In [5]:
rag_traced = RAGTraced(index=index, llm_client=client)

In [6]:
answer = rag_traced.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

It keeps calling the model inside a `while True` loop, and after each response it checks whether the model returned any `function_call` items.

- If there are function calls, your code runs the tools, appends the tool outputs to `messages`, and loops again.
- If there are no function calls, `has_function_calls` stays `False`, and the loop `break`s.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In other words, it keeps looping until the model gives a final message with no more tool calls.


Re-run the query from Q1. Which span names appear in the spans table?

- Only rag
- rag and llm
- rag, search, and llm
- search, llm, and judge

In [7]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("SELECT * FROM spans", conn)
conn.close()
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search_operation,1784563964525521811,1784563964528288900,NaN,NaN,NaN
1,llm_operation,1784563964535649191,1784563966800536057,7111.0,125.0,0.005896
2,rag_operation,1784563964525425632,1784563966803710135,NaN,NaN,NaN


## Q4 Answer: still rag, search, and llm

## Q5. Querying trace data
The traces are now in SQLite. Run one more query through the traced RAG, then query the database.

The rag span wraps everything, so its duration includes both search and llm. To see where time actually goes, exclude the rag span and compare the children.

Using SQL (or pandas), compute the total duration for each span name excluding rag. Which span type takes the most total time?

- search
- llm
- They're all about the same

In [8]:
df['duration'] = df.end_time - df.start_time

In [9]:
df

,name,start_time,end_time,input_tokens,output_tokens,cost,duration
0,search_operation,1784563964525521811,1784563964528288900,NaN,NaN,NaN,2767089
1,llm_operation,1784563964535649191,1784563966800536057,7111.0,125.0,0.005896,2264886866
2,rag_operation,1784563964525425632,1784563966803710135,NaN,NaN,NaN,2278284503


## Q5 Answer: LLM takes a much longer time than search - LLM

## Q6. Token stability across runs
Load the SQLite data with pandas. One thing a dashboard can tell you is how stable your system is. If the same query always produces the same number of input tokens, the context your RAG retrieves is consistent. If it varies a lot, something in the search may be unstable.

Run the same query from Q1 three more times (so you have 4 RAG calls total in the database). Then compute the input tokens for each llm span.

How much do the input tokens vary across these 4 runs?

- They're identical
- Within 10% of each other
- Within 50% of each other
- They vary more than 50%

In [10]:
for _ in range(3):
    answer = rag_traced.rag("How does the agentic loop keep calling the model until it stops?")

In [11]:
conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("SELECT * FROM spans", conn)
conn.close()
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search_operation,1784563964525521811,1784563964528288900,NaN,NaN,NaN
1,llm_operation,1784563964535649191,1784563966800536057,7111.0,125.0,0.005896
2,rag_operation,1784563964525425632,1784563966803710135,NaN,NaN,NaN
3,search_operation,1784564086086777298,1784564086091194116,NaN,NaN,NaN
4,llm_operation,1784564086102105742,1784564088391375908,7111.0,105.0,0.005806
5,rag_operation,1784564086086709842,1784564088395247319,NaN,NaN,NaN
6,search_operation,1784564088398394950,1784564088399926968,NaN,NaN,NaN
7,llm_operation,1784564088403278958,1784564090307912495,7111.0,99.0,0.005779
8,rag_operation,1784564088398353644,1784564090312076472,NaN,NaN,NaN
9,search_operation,1784564090315544443,1784564090317133216,NaN,NaN,NaN


## Q6 Answer: input tokens seem to be identical